# 08 · Modelo predictivo

**Nuevo bloque, agregado tras conocer los criterios de evaluación del comité (capacidad predictiva, 30% del puntaje).** No forma parte del roadmap original de 7 bloques (que respondía las 4 preguntas del Plan Maestro de forma descriptiva); este notebook construye un modelo que predice el **resultado final del partido** usando únicamente información disponible **al medio tiempo**.

**Decisión de diseño clave — por qué "al medio tiempo" y no "con el partido completo":** si usáramos métricas del partido COMPLETO (posesión total, tiros totales, xG total) para predecir el resultado de ESE MISMO partido, estaríamos usando información que ocurre al mismo tiempo o después del resultado — el modelo "haría trampa" (fuga de datos / leakage). Cortando todas las features al final del primer tiempo (period == 1), el modelo predice algo que genuinamente no ha ocurrido todavía, y el resultado es directamente útil para un cuerpo técnico tomando decisiones de ajuste al descanso (conecta con el criterio de valor operativo).

**No existe un baseline publicado por el comité** (confirmado): definimos y documentamos dos baselines propios más abajo, explícitamente, para que la mejora del modelo completo sea verificable contra algo, no solo "un número que suena bien".

In [ ]:
import sys
sys.path.append("../src")
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import log_loss, accuracy_score, r2_score, classification_report, confusion_matrix

from framework_analitico import (
    normalizar_direccion_ataque,
    dataset_prediccion_resultado,
)

EQUIPO = "América"  # cambia esto para reaplicar el modelo a otro entrenador/equipo

events = pd.read_csv("../data/raw/events_jardine_america.csv", low_memory=False)
partidos = pd.read_csv("../data/raw/partidos_jardine_america.csv")

events_norm = normalizar_direccion_ataque(events, equipo=EQUIPO)

dataset = dataset_prediccion_resultado(events_norm, equipo=EQUIPO, partidos_df=partidos)
dataset["match_date"] = pd.to_datetime(dataset["match_date"])
dataset = dataset.sort_values("match_date").reset_index(drop=True)

print(f"Partidos en el dataset: {len(dataset)}")
print(dataset["temporada"].value_counts())
print()
print(dataset["resultado_final"].value_counts())

## 1. Split cronológico (train / holdout)

**Por qué cronológico y no aleatorio:** un split aleatorio mezclaría partidos de las 3 temporadas en train y en holdout, lo cual es optimista de forma artificial — en la práctica, un modelo real se entrenaría con el pasado para predecir el futuro, nunca al revés. Usamos **temporadas 2023/2024 + 2024/2025 como train** (91 partidos) y **2025/2026 como holdout** (38 partidos, la temporada más reciente) — una validación "fuera de muestra en el tiempo", más exigente y más defendible que un split aleatorio.

In [ ]:
FEATURES = [
    "posesion_pct_1t", "field_tilt_pct_1t", "field_tilt_relativo_pct_1t", "ppda_1t",
    "tiros_favor_1t", "tiros_contra_1t", "xg_favor_1t", "xg_contra_1t",
    "pct_pases_largos_1t", "longitud_promedio_m_1t",
    "goles_favor_1t", "goles_contra_1t",
]

dataset["es_local"] = (dataset["localidad"] == "Local").astype(int)
FEATURES_LOCALIDAD = FEATURES + ["es_local"]

train = dataset[dataset["temporada"] != "2025/2026"].copy()
holdout = dataset[dataset["temporada"] == "2025/2026"].copy()
print(f"Train: {len(train)} partidos (2023/2024 + 2024/2025)")
print(f"Holdout: {len(holdout)} partidos (2025/2026)")

X_train, X_holdout = train[FEATURES_LOCALIDAD], holdout[FEATURES_LOCALIDAD]
y_train_clase, y_holdout_clase = train["resultado_final"], holdout["resultado_final"]
y_train_dif, y_holdout_dif = train["dif_gol_final"], holdout["dif_gol_final"]

# IMPORTANTE: sklearn.metrics.log_loss asume que las columnas de probabilidad
# siguen el orden LEXICOGRÁFICO de las clases, sin importar el orden que se
# le pase en `labels` — por eso CLASES se define con sorted() y no a mano.
CLASES = sorted(["Pierde", "Empata", "Gana"])
print("Orden de clases (lexicográfico, requerido por log_loss):", CLASES)

## 2. Baseline 1 — ingenuo (distribución histórica de resultados)

Predice, para TODOS los partidos del holdout, la misma probabilidad constante por clase: la frecuencia histórica de Gana/Empata/Pierde observada en el train. No usa ninguna métrica del partido — es el piso mínimo que cualquier modelo con información real debería superar.

In [ ]:
dist_train = y_train_clase.value_counts(normalize=True).reindex(CLASES).values
probs_baseline1 = np.tile(dist_train, (len(holdout), 1))

logloss_baseline1 = log_loss(y_holdout_clase, probs_baseline1, labels=CLASES)
acc_baseline1 = accuracy_score(y_holdout_clase, [CLASES[np.argmax(dist_train)]] * len(holdout))

print(f"Distribución de train ({', '.join(CLASES)}): {dist_train.round(3)}")
print(f"Baseline 1 (ingenuo) — log-loss: {logloss_baseline1:.4f} | accuracy: {acc_baseline1:.3f}")

## 3. Baseline 2 — informado, solo marcador al descanso

Un segundo baseline, más exigente: ¿cuánto log-loss se gana con SOLO saber el marcador al descanso (goles a favor/en contra en el primer tiempo), sin ninguna de las métricas tácticas del framework (posesión, PPDA, field tilt, xG)? Esto aísla cuánto aporta el framework táctico completo por encima de lo obvio.

In [ ]:
FEATURES_MARCADOR = ["goles_favor_1t", "goles_contra_1t"]

scaler_b2 = StandardScaler().fit(train[FEATURES_MARCADOR])
modelo_baseline2 = LogisticRegression(max_iter=2000, C=1.0)
modelo_baseline2.fit(scaler_b2.transform(train[FEATURES_MARCADOR]), y_train_clase)

probs_baseline2 = modelo_baseline2.predict_proba(scaler_b2.transform(holdout[FEATURES_MARCADOR]))
# reordenar columnas de predict_proba al orden fijo CLASES
orden = [list(modelo_baseline2.classes_).index(c) for c in CLASES]
probs_baseline2 = probs_baseline2[:, orden]

logloss_baseline2 = log_loss(y_holdout_clase, probs_baseline2, labels=CLASES)
preds_baseline2 = modelo_baseline2.predict(scaler_b2.transform(holdout[FEATURES_MARCADOR]))
acc_baseline2 = accuracy_score(y_holdout_clase, preds_baseline2)

print(f"Baseline 2 (solo marcador al descanso) — log-loss: {logloss_baseline2:.4f} | accuracy: {acc_baseline2:.3f}")

## 4. Modelo completo — regresión logística multinomial con todas las features del primer tiempo

Incluye posesión, field tilt, PPDA, tiros/xG a favor y en contra, % de pases largos, longitud promedio de pase, marcador al descanso y localidad — todo cortado al minuto 45. Se estandarizan las features (ajustado SOLO con train, para no filtrar información del holdout a la normalización).

In [ ]:
scaler = StandardScaler().fit(X_train)
X_train_s = scaler.transform(X_train)
X_holdout_s = scaler.transform(X_holdout)

modelo = LogisticRegression(max_iter=2000, C=1.0)
modelo.fit(X_train_s, y_train_clase)

probs_modelo = modelo.predict_proba(X_holdout_s)
orden = [list(modelo.classes_).index(c) for c in CLASES]
probs_modelo = probs_modelo[:, orden]

logloss_modelo = log_loss(y_holdout_clase, probs_modelo, labels=CLASES)
preds_modelo = modelo.predict(X_holdout_s)
acc_modelo = accuracy_score(y_holdout_clase, preds_modelo)

print(f"Modelo completo — log-loss: {logloss_modelo:.4f} | accuracy: {acc_modelo:.3f}")
print()
print(classification_report(y_holdout_clase, preds_modelo, labels=CLASES))
print()
print("Matriz de confusión (filas=real, columnas=predicho), orden", CLASES)
print(confusion_matrix(y_holdout_clase, preds_modelo, labels=CLASES))

## 5. Comparación de los 3 enfoques (clasificación)

Log-loss más bajo = mejor (probabilidades mejor calibradas con el resultado real). El modelo completo debe superar a ambos baselines para justificar el uso del framework táctico completo.

In [ ]:
comparacion_clasificacion = pd.DataFrame({
    "modelo": ["Baseline 1: ingenuo (distribución histórica)", "Baseline 2: solo marcador al descanso", "Modelo completo (features _1t)"],
    "log_loss": [logloss_baseline1, logloss_baseline2, logloss_modelo],
    "accuracy": [acc_baseline1, acc_baseline2, acc_modelo],
})
comparacion_clasificacion["mejora_vs_baseline1_pct"] = (
    (comparacion_clasificacion["log_loss"].iloc[0] - comparacion_clasificacion["log_loss"]) / comparacion_clasificacion["log_loss"].iloc[0] * 100
).round(1)
comparacion_clasificacion

## 6. Componente de regresión (R²): diferencia de goles final

El mismo ejercicio, pero prediciendo una variable continua (`dif_gol_final` = goles a favor − goles en contra al final del partido) con regresión lineal, para cubrir el componente R² del criterio de capacidad predictiva (el log-loss aplica al componente de clasificación).

In [ ]:
baseline_reg_pred = np.full(len(holdout), y_train_dif.mean())
r2_baseline_reg = r2_score(y_holdout_dif, baseline_reg_pred)

modelo_reg = LinearRegression()
modelo_reg.fit(X_train_s, y_train_dif)
preds_reg = modelo_reg.predict(X_holdout_s)
r2_modelo_reg = r2_score(y_holdout_dif, preds_reg)

comparacion_regresion = pd.DataFrame({
    "modelo": ["Baseline: predice el promedio de train", "Modelo completo (features _1t)"],
    "r2_holdout": [r2_baseline_reg, r2_modelo_reg],
})
comparacion_regresion

## 7. Calibración de probabilidades

Un modelo bien calibrado: cuando dice "70% de probabilidad de ganar", debería ganar efectivamente cerca del 70% de las veces en ese grupo. Con solo 38 partidos de holdout no hay suficientes datos para una curva de calibración fina, así que se reporta de forma agregada: la probabilidad promedio asignada a la clase que el modelo predijo como más probable, contra la accuracy real en ese mismo grupo.

In [ ]:
prob_max = probs_modelo.max(axis=1)
acierto = (preds_modelo == y_holdout_clase.values)

calibracion = pd.DataFrame({
    "confianza_promedio_del_modelo": [prob_max.mean()],
    "accuracy_real_en_esos_casos": [acierto.mean()],
    "diferencia": [prob_max.mean() - acierto.mean()],
})
print("Calibración agregada (holdout completo, n=38):")
calibracion

## 8. Síntesis y limitaciones

**El modelo completo supera a ambos baselines de forma consistente, tanto en clasificación como en regresión.** En log-loss: baseline ingenuo 1.088 → baseline de marcador al descanso 0.873 (19.8% de mejora) → modelo completo 0.842 (22.6% de mejora total). En R² (diferencia de goles): baseline -0.105 → modelo completo 0.190. El framework táctico completo (posesión, field tilt, PPDA, tiros/xG, directividad) agrega señal incremental real por encima de lo obvio (el marcador al descanso), aunque esa ganancia adicional es más modesta (2.8 puntos de log-loss) que la que ya aporta solo saber el marcador.

**El modelo discrimina bien las victorias pero es conservador prediciendo derrotas.** Recall de 0.83 en "Gana" (15 de 18 aciertos) frente a solo 0.20 en "Pierde" (2 de 10): el modelo rara vez se atreve a predecir una derrota con información de solo el primer tiempo, probablemente porque, al descanso, un partido que terminará en derrota luce táctimente parecido a uno que terminará empatado. Cuando sí predice "Pierde", acierta siempre (precisión 1.00) — es selectivo, no equivocado.

**Las probabilidades están levemente sobreconfiadas:** el modelo asigna en promedio 69.8% de confianza a su predicción más probable, pero acierta 60.5% de las veces (diferencia de 9.3 puntos). No es un desajuste severo, pero las probabilidades deben leerse con ese descuento — no tomarse de forma literal como "70% de probabilidad real".

**Limitaciones explícitas (documentadas, no resueltas):**
- Holdout de solo 38 partidos (una temporada): los números de log-loss/R²/calibración tienen incertidumbre considerable; no deben tratarse como estimaciones precisas de un modelo en producción.
- No se incluyó la fuerza del rival como feature explícita (más allá de lo que el propio desempeño táctico de América ya refleja implícitamente) por no contar con un rating externo de equipos — una extensión futura natural.
- El modelo predice el resultado a partir del desempeño EN el primer tiempo, no antes del partido: es un modelo de "ajuste al descanso", no un pronosticador pre-partido (ver nota de diseño al inicio del notebook). Esto es una decisión deliberada, no una limitación accidental, pero cambia qué pregunta de negocio responde el modelo.
- Clases desbalanceadas (Gana 54.9% del train) — el recall bajo en "Pierde" es en parte un síntoma de esto, no solo de la dificultad de la señal.

**Conexión con el criterio de valor operativo:** este modelo, tal como está diseñado (features cortadas al medio tiempo), es directamente utilizable por un cuerpo técnico al descanso — da una probabilidad de resultado final con la información disponible en ese momento, útil para decidir si ajustar o mantener el planteamiento. Esto se retomará en el bloque de "coach drills" / demo de valor operativo.